In [13]:
import pyspark
from pyspark.sql import SparkSession, types

In [14]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

print(f"Spark version: {spark.version}")

Spark version: 4.1.1


In [15]:
# !curl -o yellow_tripdata_2025-11.parquet https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet

In [16]:
df = spark.read.parquet("/Users/saif/Desktop/ZoomCamp Learning/Data-Engineering-ZoomCamp-Learning/06-Batch-Processing-Pyspark/yellow_tripdata_2025-11.parquet")

In [17]:
df.dtypes
df.show()
df.printSchema()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       7| 2025-11-01 00:13:25|  2025-11-01 00:13:25|              1|         1.68|         1|                 N|          43|    

In [18]:
df = df.repartition(4)

In [19]:
df.write.parquet('yellow_tripdata_2025-11-partitionated', mode="overwrite")

In [20]:
from pyspark.sql import functions as F

In [21]:
df \
    .withColumn('tpep_pickup_datetime', F.to_date(df.tpep_pickup_datetime)) \
    .withColumn('tpep_dropoff_datetime', F.to_date(df.tpep_dropoff_datetime)) \
    .select('VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime', 'PULocationID', 'DOLocationID') \
    .show()

+--------+--------------------+---------------------+------------+------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|PULocationID|DOLocationID|
+--------+--------------------+---------------------+------------+------------+
|       2|          2025-11-01|           2025-11-01|         137|         263|
|       2|          2025-11-02|           2025-11-02|         238|         151|
|       1|          2025-11-09|           2025-11-09|         132|         144|
|       2|          2025-11-07|           2025-11-07|         142|         137|
|       2|          2025-11-04|           2025-11-04|         234|          68|
|       2|          2025-11-06|           2025-11-06|         234|         230|
|       2|          2025-11-01|           2025-11-01|         107|         107|
|       2|          2025-11-01|           2025-11-01|          68|         170|
|       2|          2025-11-09|           2025-11-09|         166|         244|
|       2|          2025-11-03|         

In [22]:
df.select('tpep_pickup_datetime', 'tpep_dropoff_datetime', 'PULocationID', 'DOLocationID') \
.filter(df.total_amount > 200).show()

+--------------------+---------------------+------------+------------+
|tpep_pickup_datetime|tpep_dropoff_datetime|PULocationID|DOLocationID|
+--------------------+---------------------+------------+------------+
| 2025-11-07 12:29:33|  2025-11-07 13:36:17|         132|         265|
| 2025-11-07 16:21:28|  2025-11-07 16:21:33|          48|          48|
| 2025-11-04 16:53:07|  2025-11-04 18:06:47|         132|         265|
| 2025-11-09 21:52:30|  2025-11-09 22:44:13|         132|         265|
| 2025-11-07 20:50:28|  2025-11-07 21:51:22|         132|         265|
| 2025-11-06 15:12:04|  2025-11-06 15:12:11|         180|         180|
| 2025-11-03 16:43:47|  2025-11-03 18:28:13|         132|         265|
| 2025-11-08 08:40:33|  2025-11-08 09:29:12|         138|         265|
| 2025-11-03 21:38:36|  2025-11-03 22:36:27|         132|         265|
| 2025-11-10 03:22:44|  2025-11-10 04:09:24|         132|         265|
| 2025-11-02 21:26:52|  2025-11-02 22:16:02|         132|         265|
| 2025

In [23]:
df = df.withColumnRenamed('tpep_pickup_datetime', 'pickup_time') \
    .withColumnRenamed('tpep_dropoff_datetime', 'dropoff_time')
df.columns

['VendorID',
 'pickup_time',
 'dropoff_time',
 'passenger_count',
 'trip_distance',
 'RatecodeID',
 'store_and_fwd_flag',
 'PULocationID',
 'DOLocationID',
 'payment_type',
 'fare_amount',
 'extra',
 'mta_tax',
 'tip_amount',
 'tolls_amount',
 'improvement_surcharge',
 'total_amount',
 'congestion_surcharge',
 'Airport_fee',
 'cbd_congestion_fee']

In [24]:
df.registerTempTable('trips_data')

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/pyspark/sql/classic/dataframe.py:178: FutureWarning: Deprecated in 2.0, use createOrReplaceTempView instead.
  warnings.warn("Deprecated in 2.0, use createOrReplaceTempView instead.", FutureWarning)


In [25]:
spark.sql("""
SELECT
    *
FROM trips_data
    WHERE pickup_time >= '2025-11-15'
  AND pickup_time < '2025-11-16'
""").show()

+--------+-------------------+-------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|        pickup_time|       dropoff_time|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+-------------------+-------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       2|2025-11-15 21:50:52|2025-11-15 22:05:02|              5|         1.56|         1|                 N|         144|         137|   

In [26]:
spark.sql("""
SELECT
    count(*)
FROM trips_data
    WHERE pickup_time >= '2025-11-15'
  AND pickup_time < '2025-11-16'
""").show()

+--------+
|count(1)|
+--------+
|  162604|
+--------+



In [27]:
spark.sql("""
SELECT
    MAX((unix_timestamp(dropoff_time) - unix_timestamp(pickup_time)) / 3600.0) AS longest_trip_hours
FROM trips_data
""").show()

+------------------+
|longest_trip_hours|
+------------------+
|         90.646667|
+------------------+



In [28]:
# !curl -o taxi_zone_lookup.parquet https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv



In [33]:
df_zones = spark.read.option("header", "True").csv('taxi_zone_lookup.csv')
df_zones.write.parquet("taxi_zone_lookup.parquet")

In [34]:
df_joined = df.join(df_zones, df_zones.LocationID == df.PULocationID  )
df_joined.show()

+--------+-------------------+-------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+----------+---------+--------------------+------------+
|VendorID|        pickup_time|       dropoff_time|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|LocationID|  Borough|                Zone|service_zone|
+--------+-------------------+-------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+----------+---------+---------

In [58]:
df_joined.registerTempTable("result_table")

spark.sql("""

SELECT 
    Count (Zone) AS Zone_freq,
    Zone
FROM result_table
GROUP BY 2
ORDER BY 1

""").show()

+---------+--------------------+
|Zone_freq|                Zone|
+---------+--------------------+
|        1|Governor's Island...|
|        1|Eltingville/Annad...|
|        1|       Arden Heights|
|        3|       Port Richmond|
|        4|       Rikers Island|
|        4|   Rossville/Woodrow|
|        4|         Great Kills|
|        4| Green-Wood Cemetery|
|        5|         Jamaica Bay|
|       12|         Westerleigh|
|       14|New Dorp/Midland ...|
|       14|       West Brighton|
|       14|             Oakwood|
|       14|        Crotona Park|
|       15|       Willets Point|
|       16|Breezy Point/Fort...|
|       17|Saint George/New ...|
|       18|       Broad Channel|
|       21|     Mariners Harbor|
|       22|Heartland Village...|
+---------+--------------------+
only showing top 20 rows
